# BSk24 deformation experiment

This notebook is a small front end to the installed `eos_generation` package. It contains no scientific equations or solver implementation. Edit the single settings cell, then choose **Run All**.

**Default: `dataset_40`, up to 6 case workers.** This stellar-only, diagnostics-off profile uses 40 sequence points at rtol=1e-10 and atol=1e-12, retaining all-node tides, fixed-mass and maximum-mass searches, and the strict-family thermodynamic stages and evidence. It is an experimental dataset profile, not a STRICT certificate, and has no per-case stellar refinement envelope; the full multi-stage `strict` profile remains separately available and unchanged. Small batches or lower-core machines use fewer workers; nested pools remain disabled. Restart the kernel after updating. Previously saved outputs below are historical and do not describe these new settings: run a fresh passive preview with the execution flag False.

The first pass is always a passive preview: it performs zero scientific solver calls and writes nothing. Review the printed plan and destination. To run that exact plan, change only `EXECUTE_REVIEWED_PLAN` to `True` and choose **Run All** again in the same kernel. Any change to settings, source code, environment, process budget, plan, or destination invalidates the preview.

Scalar and list values are both accepted for amplitudes and deformation geometry. Lists form a Cartesian grid automatically; `EPSILON_MATCH` remains one governed matching anchor per experiment. `CALCULATION="thermodynamics"` stops after thermodynamic gating and reconstruction; `CALCULATION="stellar"` requests the governed full stellar route, including fixed-mass and tidal results. `PRECISION="quick"` changes numerical resolution only—the physical acceptance gates are identical to `"strict"`. After every successful execution, the notebook creates one flat experiment-level `plots/` folder. Every applicable standard graph combines all accepted EoSs from the current sweep; rejected cases are excluded, exact duplicate curves are drawn once, and no per-case plot directories are created there. This saved-table-only step makes zero solver calls and never changes an authoritative packet.

In [ ]:
from eos_generation.notebook import NotebookSettings, get_notebook_session

notebook_session = get_notebook_session()

## Experiment settings

All energy-density values are in MeV fm$^{-3}$ and fixed masses are in solar masses. `EPSILON_MATCH="standard"` uses the governed BSk24 matching anchor; numeric alternatives remain subject to the production model's declared-domain checks. `DIAGNOSTICS="on"` is available only for stellar calculations and adds the governed endpoint radial-support diagnostic. Generated packets are written below the ignored `runs/` directory only after explicit execution.

In [ ]:
# POSITIVE AMPLITUDES — DAYTIME
AMPLITUDES = [0.0, 0.04, 0.08, 0.12, 0.16, 0.20, 0.24, 0.28, 0.32]
EPSILON_MATCH = 80.0

CENTER = [200.0, 350.0, 500.0, 700.0, 950.0]
WIDTH = [150.0, 325.0, 500.0, 700.0, 900.0]
RAMP_WIDTH = [125.0, 175.0, 225.0, 275.0, 350.0]

CALCULATION = "stellar"
FIXED_MASSES = [1.4]
PRECISION = "dataset_40"
DIAGNOSTICS = "off"

# Preview with False; then change only this flag to True.
EXECUTE_REVIEWED_PLAN = False

In [ ]:
settings = NotebookSettings.from_values(
    amplitudes=AMPLITUDES,
    epsilon_match=EPSILON_MATCH,
    center=CENTER,
    width=WIDTH,
    ramp_width=RAMP_WIDTH,
    calculation=CALCULATION,
    fixed_masses=FIXED_MASSES,
    precision=PRECISION,
    diagnostics=DIAGNOSTICS,
)

In [ ]:
import hashlib

presentation_sources = {
    name: hashlib.sha256((notebook_session.repository_root / "notebooks" / name).read_bytes()).hexdigest()
    for name in ("eos_catalogue.py", "build_experiment_plots.py")
}
if not EXECUTE_REVIEWED_PLAN:
    reviewed_presentation_sources = presentation_sources
elif globals().get("reviewed_presentation_sources") != presentation_sources:
    raise RuntimeError("Presentation source changed or was not previewed; Run All with EXECUTE_REVIEWED_PLAN=False first.")

notebook_run = notebook_session.prepare(
    settings, record_preview=not EXECUTE_REVIEWED_PLAN
)
print(notebook_run.summary_text())

In [ ]:
experiment_result = notebook_session.execute(
    notebook_run,
    current_settings=settings,
    execute=EXECUTE_REVIEWED_PLAN,
)
if experiment_result is None:
    print("EXECUTE_REVIEWED_PLAN=False: execution remains disabled.")
else:
    from IPython.display import Markdown, display
    import json
    from pathlib import Path
    import subprocess
    import sys
    import pandas as pd

    print(f"Experiment complete: {notebook_run.output_root}")
    print(experiment_result.summary_text())
    if reviewed_presentation_sources != {
        name: hashlib.sha256((notebook_session.repository_root / "notebooks" / name).read_bytes()).hexdigest()
        for name in reviewed_presentation_sources
    }:
        raise RuntimeError("Scientific run complete, but reporting source changed during execution. Preserve the packet and rebuild reporting separately.")
    eos_data_destination = notebook_run.planning_root / "EOS_DATA"
    catalogue_builder = notebook_session.repository_root / "notebooks" / "eos_catalogue.py"
    try:
        labelled = subprocess.run(
            [sys.executable, str(catalogue_builder),
             "--repository-root", str(notebook_session.repository_root),
             "--experiment", str(notebook_run.output_root),
             "--destination", str(eos_data_destination)],
            cwd=notebook_session.repository_root, check=True, capture_output=True, text=True,
        )
        catalogue_result = json.loads(labelled.stdout)
    except (subprocess.CalledProcessError, json.JSONDecodeError) as error:
        print("Scientific run complete, but friendly EoS reporting failed. Preserve the packet; do not rerun solvers to recover labels.")
        print(getattr(error, "stderr", "") or str(error))
        raise
    aliases = pd.read_csv(eos_data_destination / "case_aliases.csv", dtype=str, keep_default_na=False)
    print(f"Friendly labels ready: {catalogue_result['unique_eos_count']} physical EoSs; 0 solver calls.")
    presentation_columns = [
        "eos_id",
        "status",
        "acceptance_domain",
        "retained_epsilon_max_mev_fm3",
        "retained_pressure_max_mev_fm3",
        "retained_endpoint_reason",
        "requested_fixed_masses_status",
        "maximum_mass_availability_status",
        "student_view_eligibility_status",
    ]
    for geometry_index in range(1, len(experiment_result.child_results) + 1):
        ledger = experiment_result.table("case_ledger.csv", geometry_index=geometry_index)
        geometry_aliases = aliases.loc[aliases.geometry_id.eq(f"geometry_{geometry_index:03d}")]
        lookup = geometry_aliases.set_index("source_case_id")["eos_id"]
        ledger.insert(0, "eos_id", ledger.case_id.map(lookup).fillna(""))
        visible = [name for name in presentation_columns if name in ledger.columns]
        display(Markdown(f"## Geometry {geometry_index}: retained domains and availability"))
        display(ledger.loc[:, visible])
    combined_locations = {}

    combined_destination = notebook_run.planning_root / "plots"
    builder = notebook_session.repository_root / "notebooks" / "build_experiment_plots.py"
    command = [
        sys.executable,
        str(builder),
        "--repository-root",
        str(notebook_session.repository_root),
        "--experiment",
        str(notebook_run.output_root),
        "--destination",
        str(combined_destination),
        "--eos-data",
        str(eos_data_destination),
    ]
    try:
        completed = subprocess.run(
            command,
            cwd=notebook_session.repository_root,
            check=True,
            capture_output=True,
            text=True,
        )
        combined_result = json.loads(completed.stdout)
        combined_locations = {
            "Combined accepted plots": Path(combined_result["plots_path"]),
        }
        print(
            "Combined accepted plot folder complete: "
            f"{combined_result['generated_plot_count']} plots from "
            f"{combined_result['accepted_case_occurrence_count']} accepted cases; "
            f"{combined_result['excluded_rejected_case_occurrence_count']} rejected cases excluded; "
            "0 solver calls."
        )
    except (subprocess.CalledProcessError, json.JSONDecodeError, KeyError) as error:
        print("Scientific run complete, but combined accepted plotting failed.")
        details = getattr(error, "stderr", "") or str(error)
        print(str(details).strip())
    locations = notebook_session.student_view_locations()
    locations.update({
        "Labelled EoS tables": eos_data_destination,
        "Friendly EoS catalogue": eos_data_destination / "eos_catalogue.csv",
        "Original case ID mapping": eos_data_destination / "case_aliases.csv",
        "Shared persistent numbering (archive this)": Path(catalogue_result["catalogue_path"]),
    })
    locations.update(combined_locations)
    links = []
    for label, path in locations.items():
        relative = "../" + path.relative_to(notebook_session.repository_root).as_posix()
        links.append(f"- [{label}]({relative})")
    display(Markdown("## Student result locations\n\n" + "\n".join(links)))

## Interpreting a completed run

Accepted, rejected, and unresolved cases retain their complete raw evidence and exact status. The first continuously resolved $c_s^2=1$ crossing is included as the case-specific endpoint; any later return below one remains outside the usable branch. A rejected or unresolved raw proposal receives no reconstruction or stellar calculation. Hard scientific validity is reported separately from observable availability, and finite auxiliary diagnostics do not by themselves reject a case. The reconstructed state is an effective one-fluid cold barotrope; it does not establish microscopic composition or beta equilibrium. Fixed-mass observables require a true bracket inside the retained domain. They remain available when valid even if the endpoint prevents maximum-mass resolution; maximum mass is resolved only when a turning point is bracketed and refined under the selected numerical profile.

After a successful run, open **Student result locations > Read me first** for the saved-table hierarchy and **Combined accepted plots** for the single flat graph folder. The data remain organized as `experiment -> geometry_NNN -> case_id -> sampled row`, but every PNG in `plots/` overlays the applicable accepted EoSs from this current sweep. Exact duplicate curves are drawn once, rejected cases are excluded, failed stellar gaps are never bridged, and plotting never reruns a solver. Separate packet-level figures remain only as sealed technical evidence.

**Friendly EoS catalogue** and **Labelled EoS tables** add persistent labels such as `H000001` without changing canonical case IDs. Numbering continues across positive and negative runs using the append-only `runs/eos_catalogue/` registry; archive that directory with your data and never reset it. `H000000` is the first registered baseline; repeated identity controls share its physical label. Identical physical definitions reuse labels across QUICK/STRICT evaluations, while numerical stages, provenance, failures and missing values remain separate. Rejected proposals have no accepted-EoS label. These derived tables do not replace the byte-for-byte `STUDENT_VIEW` copies or sealed packets. Dense plots use colour bars and the catalogue mapping rather than unreadable legends containing thousands of IDs.